# InclusionScore AI -- Alternate Credit Scoring

**HackerEarth Hack-o-Hire 2026 | Theme 4: AI-Powered Alternate Credit Scoring**

This notebook is a **single, linear pipeline** that reproduces the entire project end-to-end:

| Step | Section |
|------|---------|
| 1 | Data Loading & EDA |
| 2 | Data Cleaning & Feature Engineering |
| 3 | Train / Validation / Test Split |
| 4 | Model Training (Logistic Regression baseline + XGBoost) |
| 5 | Evaluation (AUC-ROC, Precision-Recall, Confusion Matrix, Calibration) |
| 6 | SHAP Explainability (Global + Local) |
| 7 | Fairness Audit (Disparate Impact, Demographic Parity) |
| 8 | Live Scoring Demo |

**Dataset:** [Give Me Some Credit](https://www.kaggle.com/c/GiveMeSomeCredit) (150k borrowers, 10 features, binary default target)
**Goal:** Predict probability of serious delinquency (90+ days past due) within 2 years, while ensuring model transparency and fairness.

In [ ]:
import sys, os, json, warnings
sys.path.insert(0, os.path.abspath('..'))
warnings.filterwarnings('ignore', category=FutureWarning)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap
from sklearn.metrics import (
    roc_curve, precision_recall_curve, confusion_matrix,
    roc_auc_score, brier_score_loss,
)
from sklearn.calibration import calibration_curve

from src.data_loader import load_primary_dataset
from src.utils import summarize_df
from src.features import clean_df, create_features, TARGET_COL
from src.models import (
    train_baseline, train_xgboost, evaluate,
    save_model, load_model,
)
from src.config import RANDOM_SEED, SCORE_THRESHOLDS

sns.set_theme(style='whitegrid')
%matplotlib inline
print('All imports OK.')

---
## 1. Data Loading & Exploratory Data Analysis

Load the raw *Give Me Some Credit* CSV and understand its shape, distributions, missing values, and anomalies.

In [ ]:
df_raw = load_primary_dataset()
print(f'Shape: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns')
df_raw.head()

In [ ]:
df_raw.info()
print()
df_raw.describe()

### 1.1 Target Distribution

`SeriousDlqin2yrs` = 1 means the borrower experienced 90+ days past due within 2 years.

In [ ]:
counts = df_raw['SeriousDlqin2yrs'].value_counts()
print(f'No Default (0): {counts[0]:,} ({counts[0]/len(df_raw)*100:.1f}%)')
print(f'Default    (1): {counts[1]:,} ({counts[1]/len(df_raw)*100:.1f}%)')
print(f'Imbalance ratio: 1:{counts[0]//counts[1]}')

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(['No Default (0)', 'Default (1)'], counts.values, color=['#2196F3', '#f44336'])
ax.set_ylabel('Count')
ax.set_title('Target Distribution: SeriousDlqin2yrs')
for i, v in enumerate(counts.values):
    ax.text(i, v + 1000, f'{v:,} ({v/len(df_raw)*100:.1f}%)', ha='center', fontsize=10)
plt.tight_layout()
plt.show()

### 1.2 Missing Values

In [ ]:
null_counts = df_raw.isnull().sum()
null_pct = (null_counts / len(df_raw) * 100).round(2)
missing_df = pd.DataFrame({'count': null_counts, 'pct': null_pct}).sort_values('count', ascending=False)
print(missing_df[missing_df['count'] > 0])

fig, ax = plt.subplots(figsize=(8, 3))
cols_with_nulls = null_pct[null_pct > 0].sort_values(ascending=False)
ax.barh(cols_with_nulls.index, cols_with_nulls.values, color='#FF9800')
ax.set_xlabel('Missing (%)')
ax.set_title('Missing Values by Column')
for i, v in enumerate(cols_with_nulls.values):
    ax.text(v + 0.3, i, f'{v:.1f}%', va='center', fontsize=9)
plt.tight_layout()
plt.show()

### 1.3 Correlation Matrix

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
corr = df_raw.corr(numeric_only=True)
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, square=True, linewidths=0.5, ax=ax, cbar_kws={'shrink': 0.8})
ax.set_title('Correlation Matrix')
plt.tight_layout()
plt.show()

print('\nCorrelation with target:')
print(corr['SeriousDlqin2yrs'].drop('SeriousDlqin2yrs').sort_values(ascending=False))

### 1.4 Feature Distributions by Default Status

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
df_raw[df_raw['SeriousDlqin2yrs']==0]['age'].hist(bins=50, alpha=0.6, label='No Default', color='#2196F3', ax=ax, density=True)
df_raw[df_raw['SeriousDlqin2yrs']==1]['age'].hist(bins=50, alpha=0.6, label='Default', color='#f44336', ax=ax, density=True)
ax.set_xlabel('Age')
ax.set_ylabel('Density')
ax.set_title('Age Distribution by Default Status')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
features_to_plot = [
    'RevolvingUtilizationOfUnsecuredLines', 'DebtRatio', 'MonthlyIncome',
    'NumberOfTime30-59DaysPastDueNotWorse', 'NumberOfTimes90DaysLate',
    'NumberOfTime60-89DaysPastDueNotWorse',
]
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for idx, feat in enumerate(features_to_plot):
    ax = axes[idx // 3][idx % 3]
    p99 = df_raw[feat].quantile(0.99)
    clipped = df_raw[feat].clip(upper=p99)
    sns.boxplot(x=df_raw['SeriousDlqin2yrs'], y=clipped, hue=df_raw['SeriousDlqin2yrs'],
                palette=['#2196F3', '#f44336'], ax=ax, legend=False)
    ax.set_title(feat[:30], fontsize=9)
    ax.set_xlabel('Default')
plt.suptitle('Feature Distributions by Default (clipped at 99th pctile)', fontsize=12)
plt.tight_layout()
plt.show()

### 1.5 Delinquency Sentinel Values

The three past-due columns contain suspicious values of **96 and 98** -- likely data-entry sentinels, not real counts.

In [ ]:
delinq_cols = [
    'NumberOfTime30-59DaysPastDueNotWorse',
    'NumberOfTime60-89DaysPastDueNotWorse',
    'NumberOfTimes90DaysLate'
]
for col in delinq_cols:
    sentinel = df_raw[col].isin([96, 98]).sum()
    print(f'{col}: {sentinel} rows with value 96 or 98')

print(f'\nage == 0: {(df_raw.age == 0).sum()} row(s) -- invalid, will be dropped')
print(f'RevolvingUtilization > 1: {(df_raw.RevolvingUtilizationOfUnsecuredLines > 1).sum()} rows -- will cap at 1.0')

---
## 2. Data Cleaning & Feature Engineering

| Step | Action |
|------|--------|
| Drop rows | `age == 0` (invalid) |
| Impute `MonthlyIncome` | Median imputation (19.8% missing) |
| Impute `NumberOfDependents` | Mode imputation (2.6% missing, mode = 0) |
| Cap `RevolvingUtilization` | Clip at 1.0 |
| Cap `DebtRatio` | Clip at 99th percentile |
| Cap delinquency sentinels | Replace 96/98 with 15 |
| Engineered features | TotalDelinquencies, HasDelinquency, IncomeToDebtRatio, OpenCreditPerDependent, AgeBin, CreditUtilBucket |
| **No protected attributes** | Dataset has no gender, ethnicity, or religion columns |

In [ ]:
# Apply the same cleaning + feature engineering used in the modular pipeline
df_clean = clean_df(df_raw, fit=True)
df_feat  = create_features(df_clean)

print(f'After cleaning:    {df_clean.shape}')
print(f'After engineering: {df_feat.shape}')
print(f'Missing values remaining: {df_feat.isnull().sum().sum()}')
print(f'\nNew features: {[c for c in df_feat.columns if c not in df_clean.columns]}')
df_feat.head()

---
## 3. Stratified Train / Validation / Test Split (70 / 15 / 15)

We use stratified splitting to preserve the ~6.7% default rate in every fold.

In [ ]:
from sklearn.model_selection import train_test_split

y = df_feat.pop(TARGET_COL)
X = df_feat

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=RANDOM_SEED)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=RANDOM_SEED)

for name, xs, ys in [('Train', X_train, y_train), ('Val', X_val, y_val), ('Test', X_test, y_test)]:
    print(f'{name:5s}: {xs.shape[0]:>6,} rows  |  default rate = {ys.mean():.4f}')

---
## 4. Model Training

### 4.1 Baseline -- Logistic Regression (balanced class weights)

In [ ]:
lr_model, lr_val_metrics = train_baseline(X_train, y_train, X_val, y_val)
lr_test_metrics = evaluate(lr_model, X_test, y_test, label='LR (test)')

### 4.2 Primary Model -- XGBoost

Key hyperparameters: `scale_pos_weight=14` (handles 1:14 class imbalance), `max_depth=5`, `learning_rate=0.05`, `n_estimators=500`.

In [ ]:
xgb_model, xgb_val_metrics = train_xgboost(X_train, y_train, X_val, y_val)
xgb_test_metrics = evaluate(xgb_model, X_test, y_test, label='XGBoost (test)')

---
## 5. Evaluation

### 5.1 Metric Comparison

In [ ]:
comparison = pd.DataFrame({'Logistic Regression': lr_test_metrics, 'XGBoost': xgb_test_metrics}).T
comparison

### 5.2 ROC Curve

In [ ]:
y_proba_xgb = xgb_model.predict_proba(X_test)[:, 1]
y_proba_lr  = lr_model.predict_proba(X_test)[:, 1]

fig, ax = plt.subplots(figsize=(7, 5))
for name, proba, style in [('XGBoost', y_proba_xgb, '-'), ('Logistic Regression', y_proba_lr, '--')]:
    fpr, tpr, _ = roc_curve(y_test, proba)
    auc_val = roc_auc_score(y_test, proba)
    ax.plot(fpr, tpr, style, label=f'{name} (AUC={auc_val:.3f})')
ax.plot([0, 1], [0, 1], 'k:', alpha=0.5)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve (Test Set)')
ax.legend()
plt.tight_layout()
plt.show()

### 5.3 Precision-Recall Curve

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
for name, proba, style in [('XGBoost', y_proba_xgb, '-'), ('Logistic Regression', y_proba_lr, '--')]:
    prec, rec, _ = precision_recall_curve(y_test, proba)
    ax.plot(rec, prec, style, label=name)
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('Precision-Recall Curve (Test Set)')
ax.legend()
plt.tight_layout()
plt.show()

### 5.4 Confusion Matrix

In [ ]:
y_pred_xgb = xgb_model.predict(X_test)

fig, ax = plt.subplots(figsize=(6, 5))
cm = confusion_matrix(y_test, y_pred_xgb)
sns.heatmap(cm, annot=True, fmt=',d', cmap='Blues', ax=ax,
            xticklabels=['No Default', 'Default'],
            yticklabels=['No Default', 'Default'])
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title('Confusion Matrix -- XGBoost (Test Set)')
plt.tight_layout()
plt.show()

### 5.5 Calibration Curve

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
for name, proba in [('XGBoost', y_proba_xgb), ('Logistic Regression', y_proba_lr)]:
    prob_true, prob_pred = calibration_curve(y_test, proba, n_bins=10, strategy='uniform')
    ax.plot(prob_pred, prob_true, 's-', label=name)
ax.plot([0, 1], [0, 1], 'k:', alpha=0.5, label='Perfectly calibrated')
ax.set_xlabel('Mean Predicted Probability')
ax.set_ylabel('Fraction of Positives')
ax.set_title('Calibration Curve (Test Set)')
ax.legend()
plt.tight_layout()
plt.show()

print(f'Brier Score -- XGBoost: {brier_score_loss(y_test, y_proba_xgb):.4f}')
print(f'Brier Score -- LR:      {brier_score_loss(y_test, y_proba_lr):.4f}')

### 5.6 Feature Importance (XGBoost)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
importances = pd.Series(xgb_model.feature_importances_, index=X_test.columns)
importances.sort_values().plot(kind='barh', ax=ax, color='#2196F3')
ax.set_title('XGBoost Feature Importance (gain)')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.show()

---
## 6. SHAP Explainability

SHAP (SHapley Additive exPlanations) uses game-theory to attribute each feature's contribution to every individual prediction.

### 6.1 Global SHAP Summary

In [ ]:
# Compute SHAP values on a sample of the test set (for speed)
X_shap = X_test.sample(min(2000, len(X_test)), random_state=RANDOM_SEED)
explainer = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_shap)

print(f'SHAP values shape: {shap_values.shape}')

In [ ]:
# Global feature importance (mean |SHAP|)
plt.figure(figsize=(10, 7))
shap.summary_plot(shap_values, X_shap, plot_type='bar', show=False)
plt.title('SHAP Global Feature Importance')
plt.tight_layout()
plt.show()

In [ ]:
# Beeswarm plot (shows direction of effect per feature)
plt.figure(figsize=(10, 7))
shap.summary_plot(shap_values, X_shap, show=False)
plt.title('SHAP Summary (Beeswarm)')
plt.tight_layout()
plt.show()

### 6.2 Local SHAP Explanations (Individual Predictions)

Waterfall plots for 3 sample applicants showing which features push the prediction up or down.

In [ ]:
base_value = float(explainer.expected_value)

for i in range(3):
    row = X_test.iloc[[i]]
    sv = explainer.shap_values(row)[0]
    pred = float(xgb_model.predict_proba(row)[0, 1])
    print(f'\nSample {i}: predicted default prob = {pred:.3f}')

    explanation = shap.Explanation(
        values=sv,
        base_values=base_value,
        data=row.values[0],
        feature_names=list(X_test.columns),
    )
    plt.figure(figsize=(10, 6))
    shap.plots.waterfall(explanation, show=False)
    plt.title(f'Local Explanation -- Sample {i} (pred={pred:.3f})')
    plt.tight_layout()
    plt.show()

---
## 7. Fairness Audit

Since the dataset contains **no explicit protected attributes** (gender, ethnicity, religion), we use **age groups** as proxy subgroups. Age is protected under ECOA (Equal Credit Opportunity Act).

- **Approval** = predicted default probability <= 0.3
- **Disparate Impact threshold (4/5 rule)**: 0.80

In [ ]:
from src.fairness import _bin_age, compute_subgroup_metrics, compute_disparate_impact, compute_demographic_parity

groups = _bin_age(X_test['age'])
subgroup_df = compute_subgroup_metrics(y_test, y_proba_xgb, groups)

print('=== Subgroup Metrics ===')
print(subgroup_df.to_string(index=False))

In [ ]:
di = compute_disparate_impact(subgroup_df)
dp = compute_demographic_parity(subgroup_df)

print('=== Disparate Impact (4/5 Rule) ===')
print(f"Best group:  {di['best_group']} (approval rate {di['best_approval_rate']:.2%})")
print(f"Worst group: {di['worst_group']} (approval rate {di['worst_approval_rate']:.2%})")
print(f"DI ratio:    {di['di_ratio']:.4f}  {'PASS' if di['pass'] else 'FAIL'}")

print('\n=== Demographic Parity ===')
print(f"Parity gap:  {dp['parity_gap']:.4f}  {'PASS' if dp['pass'] else 'FAIL'}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Subgroup AUC
valid = subgroup_df.dropna(subset=['auc'])
axes[0].bar(valid['group'], valid['auc'], color='#2196F3')
axes[0].set_ylim(0.5, 1.0)
axes[0].set_ylabel('AUC-ROC')
axes[0].set_title('Subgroup AUC by Age Group')
for i, row in valid.iterrows():
    axes[0].text(i, row['auc'] + 0.005, f'{row["auc"]:.3f}', ha='center', fontsize=9)

# Approval rates
colors = ['#f44336' if r < di['best_approval_rate'] * 0.8 else '#4CAF50' for r in subgroup_df['approval_rate']]
axes[1].bar(subgroup_df['group'], subgroup_df['approval_rate'], color=colors)
axes[1].axhline(y=di['best_approval_rate'] * 0.8, color='red', linestyle='--', label='4/5 threshold')
axes[1].set_ylabel('Approval Rate')
axes[1].set_title('Approval Rate by Age Group')
axes[1].legend()
for i, row in subgroup_df.iterrows():
    axes[1].text(i, row['approval_rate'] + 0.01, f'{row["approval_rate"]:.2%}', ha='center', fontsize=9)

plt.tight_layout()
plt.show()

### Mitigation Recommendations

The DI ratio is below 0.80, meaning younger borrowers are disproportionately rejected. Recommended mitigations:

1. **Re-calibrate thresholds** per subgroup to equalise approval rates
2. **Apply in-processing** techniques (fairness-aware regularisation)
3. **Post-processing** adjustments: shift predicted probabilities per group so approval rates satisfy the 4/5 rule
4. **Collect additional features** (e.g., utility payments, rent history) that reduce reliance on age as a proxy

---
## 8. Live Scoring Demo

Demonstrate the scoring pipeline on a single applicant -- the same flow used by the FastAPI `/score` endpoint.

In [ ]:
from src.api import ApplicantInput, prepare_input, score

sample_applicant = ApplicantInput(
    RevolvingUtilizationOfUnsecuredLines=0.35,
    age=42,
    NumberOfTime30_59DaysPastDueNotWorse=1,
    DebtRatio=0.25,
    MonthlyIncome=5500.0,
    NumberOfOpenCreditLinesAndLoans=8,
    NumberOfTimes90DaysLate=0,
    NumberRealEstateLoansOrLines=1,
    NumberOfTime60_89DaysPastDueNotWorse=0,
    NumberOfDependents=2,
)

result = score(sample_applicant)

print('=== Scoring Result ===')
print(f'Model Version : {result.model_version}')
print(f'Default Prob  : {result.score:.4f}')
print(f'Decision      : {result.decision}')
print(f'\nTop Contributing Features:')
for f in result.top_features:
    print(f'  {f.feature:40s}  value={f.value:>10.3f}  importance={f.contribution:.4f}')

### Decision Thresholds

| Predicted Default Prob | Decision |
|------------------------|----------|
| <= 0.30 | **APPROVE** |
| 0.30 -- 0.60 | **REVIEW** (manual underwriter check) |
| > 0.60 | **REJECT** |

---
## Conclusion

### What We Built

**InclusionScore AI** is an end-to-end credit scoring system designed for unbanked and under-banked populations:

- **XGBoost model** trained on 105k borrowers achieving AUC-ROC ~0.86
- **No protected attributes** used -- the model relies only on financial behaviour features
- **6 engineered features** (TotalDelinquencies, HasDelinquency, IncomeToDebtRatio, OpenCreditPerDependent, AgeBin, CreditUtilBucket)
- **SHAP explainability** provides global feature importance and per-applicant decision breakdowns
- **Fairness audit** quantifies disparate impact and demographic parity across age groups
- **REST API** (FastAPI) serves real-time scoring with bearer-token authentication
- **Docker-ready** for production deployment

### Architecture

```
Raw Data --> clean_df() --> create_features() --> XGBoost --> score + SHAP explanation
                                                    |
                                              FastAPI /score
                                                    |
                                        { score, decision, top_features }
```

### Key Metrics

| Metric | XGBoost |
|--------|--------|
| AUC-ROC | ~0.860 |
| Precision | ~0.232 |
| Recall | ~0.737 |
| F1-Score | ~0.352 |
| Brier Score | ~0.128 |

### Fairness Status

| Metric | Value | Status |
|--------|-------|--------|
| Disparate Impact (age groups) | 0.44 | FAIL (below 0.80) |
| Demographic Parity gap | 0.48 | FAIL (above 0.15) |

Mitigation strategies (threshold re-calibration, post-processing) are documented for production deployment.